# EDA (leakage-safe 버전)

기존 노트북에서 발견된 3가지 문제를 고쳤습니다.

1. **`pitcher_id`/`batter_id` 그룹평균으로 결측치를 채우면 미래 정보가 과거 행으로 역류합니다.** → train(과거) 구간에서 계산한 **전역 median**으로만 채우고, 결측 여부 자체를 콜드스타트 플래그로 남깁니다.
2. **타겟(`control_success`)까지 스케일링 대상에 포함되어 있었습니다.** → 스케일링 자체를 제거했습니다 (상관계수는 선형변환에 불변이라 분석에 불필요하고, 트리 모델은 스케일 영향을 받지 않습니다).
3. **`LabelEncoder`는 test에서 새 카테고리가 나오면 에러가 납니다.** → `OrdinalEncoder(handle_unknown="use_encoded_value")`로 교체하고, **train에만 `fit`** 합니다.

검증 분할은 베이스라인과 동일하게 **2024시즌을 홀드아웃**으로 씁니다 — 대치/인코딩 통계도 전부 `tr`(2019~2023)에서만 계산해서 `val`(2024)에는 `transform`만 적용합니다. 이렇게 해야 검증 점수가 실제 리더보드 상황(미래를 모르는 상태에서 예측)과 같은 조건이 됩니다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

ID = "row_id"
TARGET = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]
DATA_DIR = "./data"

## 1. 데이터 로드 & 시간 기준 분할

`season == 2024`를 검증셋으로 떼어둡니다 (베이스라인과 동일). 이후 모든 통계(median, 인코딩)는 `tr`에서만 계산합니다.

In [1]:
train = pd.read_csv(f"{DATA_DIR}/train.csv", encoding="utf-8-sig")

is_val = train["season"] == 2024
tr = train.loc[~is_val].copy()
val = train.loc[is_val].copy()
print("tr:", tr.shape, "| val:", val.shape)
print(f"tr 제구성공률: {tr[TARGET].mean():.4f} | val 제구성공률: {val[TARGET].mean():.4f}")

tr: (1221585, 49) | val: (253507, 49)
tr 제구성공률: 0.5310 | val 제구성공률: 0.4861


## 2. 결측 컬럼 확인 & 콜드스타트 플래그

결측은 전부 `asof_*` 계열입니다 — 해당 투구 이전에 참고할 과거 이력이 아예 없는(커리어/시즌 첫 등장) 경우입니다. **대치하기 전에 결측 여부 자체를 이진 플래그로 남겨서**, 콜드스타트라는 정보가 사라지지 않게 합니다 (지난 EDA에서 `asof_pitcher_n`이 작을수록 성공률이 다르게 나타나는 패턴을 확인했었죠 — 이 신호를 보존하려는 목적입니다).

In [2]:
NA_COLS = [c for c in train.columns if train[c].isna().any()]
print(f"결측 있는 컬럼 {len(NA_COLS)}개:")
for c in NA_COLS:
    print(" -", c)

결측 있는 컬럼 16개:
 - asof_pitcher_success_rate
 - asof_pitcher_reverse_rate
 - asof_pitcher_middle_rate
 - asof_pitcher_ball_rate
 - asof_pitcher_strike_rate
 - asof_pitcher_prev1_game_success_rate
 - asof_pitcher_prev3_game_success_rate
 - asof_pitcher_prev5_game_success_rate
 - asof_pitcher_prev1_game_middle_rate
 - asof_pitcher_prev3_game_middle_rate
 - asof_pitcher_prev5_game_middle_rate
 - asof_batter_success_rate
 - asof_batter_middle_rate
 - asof_pitcher_fastball_rate
 - asof_pitcher_breaking_rate
 - asof_pitcher_offspeed_rate


In [3]:
for df in (tr, val):
    for c in NA_COLS:
        df[f"{c}_is_na"] = df[c].isna().astype(int)

# 참고: 위 16개 플래그는 사실상 "그 투수/타자의 첫 등장이냐 아니냐" 하나로 거의 겹칩니다
# (아래 4번 다중공선성 체크에서 그대로 확인됩니다). 실전에서는
# pitcher_is_cold_start / batter_is_cold_start 2개 정도로 합쳐 써도 충분합니다.

## 3. (진단) 왜 그룹평균 대치가 위험한가

`asof_pitcher_prev5_game_success_rate` 결측행(그 투수의 직전 5경기 기록이 없는 경우)을 두 가지 방식으로 채워서 비교합니다.

- **leaky**: `pitcher_id` 그룹평균 (tr 전체 — 즉 미래 시즌 데이터까지 포함해서 계산)
- **safe**: tr 전체의 전역 median (상수값, 개별 투수 정보 없음)

결측이었던 행만 떼서 '대치값이 실제 타겟과 얼마나 가까운가'를 보면, leaky 쪽만 유의미한 상관을 보입니다 — 이건 그 투수의 **미래 성적**이 대치값에 흘러들어간 결과이지, 예측 시점에 실제로 알 수 있는 정보가 아닙니다.

In [4]:
col = "asof_pitcher_prev5_game_success_rate"
mask = tr[col].isna()

leaky = tr[col].fillna(tr.groupby("pitcher_id")[col].transform("mean")).fillna(tr[col].median())
safe  = tr[col].fillna(tr[col].median())

print(f"결측행 수: {mask.sum()} / {len(tr)}")
print(f"전체 corr(대치후 vs 타겟)   - leaky: {np.corrcoef(leaky, tr[TARGET])[0,1]:.4f} "
      f"| safe: {np.corrcoef(safe, tr[TARGET])[0,1]:.4f}")
print(f"결측행만 corr(대치값 vs 타겟) - leaky: {np.corrcoef(leaky[mask], tr.loc[mask, TARGET])[0,1]:.4f} "
      f"| safe: 0 (상수라 상관계수 정의 안됨)")

결측행 수: 25547 / 1221585
전체 corr(대치후 vs 타겟)   - leaky: 0.0816 | safe: 0.0793
결측행만 corr(대치값 vs 타겟) - leaky: 0.1400 | safe: 0 (상수라 상관계수 정의 안됨)


결측이었던 행만 놓고 보면 leaky 대치값이 타겟과 0.14 상관을 보입니다. 이 값은 원래 그 시점엔 존재하지 않았어야 할 정보(그 투수의 미래 성적)가 새어 들어간 결과라, **로컬 검증 점수를 실제보다 좋게 보이게 만드는** 전형적인 누수 패턴입니다.

## 4. 안전한 결측치 대치

`tr`에서 계산한 전역 median으로만 채우고, 같은 값을 `val`에도 그대로 적용합니다 (median은 `tr`에서만 계산 → `val`은 `transform`만).

In [5]:
medians = tr[NA_COLS].median()

for df in (tr, val):
    df[NA_COLS] = df[NA_COLS].fillna(medians)

print("남은 결측 (tr, val):", tr[NA_COLS].isna().sum().sum(), val[NA_COLS].isna().sum().sum())

남은 결측 (tr, val): 0 0


## 5. 범주형 인코딩

`OrdinalEncoder`를 `tr`에만 `fit`합니다. `val`(그리고 나중에 test)은 `transform`만 — 새 카테고리가 나와도 `unknown_value=-1`로 처리되어 에러 없이 넘어갑니다.

In [6]:
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
enc.fit(tr[CAT_COLS])

tr[CAT_COLS] = enc.transform(tr[CAT_COLS])
val[CAT_COLS] = enc.transform(val[CAT_COLS])
print("인코딩 완료:", CAT_COLS)

인코딩 완료: ['top_bottom', 'game_type', 'base_state']


## 6. 상관관계 확인

타겟까지 포함해서 봅니다 (지난 실수 수정). 스케일링은 상관계수에 영향을 주지 않으므로 생략했습니다.

In [7]:
feat_cols = [c for c in tr.columns if c not in [ID, TARGET]]

target_corr = tr[feat_cols + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("=== 타겟과 상관관계 상위 15 ===")
print(target_corr.head(15).round(4))

=== 타겟과 상관관계 상위 15 ===
asof_pitcher_success_rate               0.0836
asof_pitcher_prev5_game_success_rate    0.0793
asof_pitcher_reverse_rate               0.0786
asof_pitcher_prev3_game_success_rate    0.0765
game_type                               0.0727
asof_pitcher_prev1_game_success_rate    0.0618
asof_batter_success_rate                0.0605
asof_batter_n                           0.0382
season                                  0.0375
asof_pitcher_middle_rate                0.0333
asof_batter_middle_rate                 0.0313
asof_pitcher_prev5_game_middle_rate     0.0268
asof_pitcher_prev3_game_middle_rate     0.0234
asof_pitcher_prev1_game_middle_rate     0.0161
game_month                              0.0150
Name: control_success, dtype: float64


In [8]:
fcorr = tr[feat_cols].corr().abs()
upper = fcorr.where(np.triu(np.ones(fcorr.shape), k=1).astype(bool))
pairs = upper.stack().reset_index()
pairs.columns = ["feature_1", "feature_2", "corr"]
pairs = pairs[pairs["corr"] >= 0.5].sort_values("corr", ascending=False)

print("=== 다중공선성 (|corr| >= 0.5) 상위 10 ===")
print(pairs.head(10).to_string(index=False))

=== 다중공선성 (|corr| >= 0.5) 상위 10 ===
                       feature_1                         feature_2  corr
asof_pitcher_success_rate_is_na  asof_pitcher_reverse_rate_is_na   1.0
 asof_pitcher_success_rate_is_na   asof_pitcher_strike_rate_is_na   1.0
 asof_pitcher_success_rate_is_na     asof_pitcher_ball_rate_is_na   1.0
 asof_pitcher_success_rate_is_na   asof_pitcher_middle_rate_is_na   1.0
 asof_pitcher_success_rate_is_na asof_pitcher_fastball_rate_is_na   1.0
 asof_pitcher_reverse_rate_is_na     asof_pitcher_ball_rate_is_na   1.0
 asof_pitcher_reverse_rate_is_na   asof_pitcher_strike_rate_is_na   1.0
 asof_pitcher_reverse_rate_is_na asof_pitcher_fastball_rate_is_na   1.0
 asof_pitcher_reverse_rate_is_na asof_pitcher_breaking_rate_is_na   1.0
 asof_pitcher_reverse_rate_is_na   asof_pitcher_middle_rate_is_na   1.0


결과 해석:

- **타겟 상관 상위권은 여전히 `asof_pitcher_*` 계열**입니다 (`asof_pitcher_success_rate` 0.084가 최고). 개별 투구 단위라 전체적으로 상관이 크진 않지만, 지난 EDA의 분위별 캘리브레이션 곡선을 보면 실질적인 예측력은 분명합니다.
- `game_type`이 0.073으로 예상보다 상위권에 있습니다 — `F`/`R` 구분이 실제로 타겟과 관련 있다는 뜻인데, test(2025, 실제 정규시즌)에 `F` 유형이 얼마나/어떻게 존재하는지 꼭 확인이 필요합니다.
- **`_is_na` 플래그끼리 상관 1.0인 쌍이 대부분**입니다 — 예상대로 같은 행(그 투수/타자의 첫 등장)에서 한꺼번에 결측이 발생하기 때문입니다. 16개를 다 넣기보다 `pitcher_is_cold_start`, `batter_is_cold_start` 2개 정도로 정리해서 쓰는 걸 추천합니다.

## 다음 단계 제안
1. `_is_na` 플래그 16개 → 2개(투수/타자 콜드스타트)로 통합
2. `game_type`(F/R)이 test에도 존재하는지, 존재한다면 비율이 train과 비슷한지 확인
3. 이 전처리를 `sklearn.pipeline.Pipeline` + `ColumnTransformer`로 감싸서, train fold에만 `fit`하고 val/test엔 `transform`만 적용되도록 강제 (지금처럼 수동으로 tr/val 나눠 처리하면 실수하기 쉬움)

---
# 7. 이어서: 콜드스타트 플래그 통합 + 실제 효과 검증

앞서 제안한 3가지 중 2개를 진행합니다.

1. `_is_na` 플래그 16개 → `pitcher_is_cold_start` / `batter_is_cold_start` 2개로 통합
2. `game_type`(F/R)이 test에도 있는지 확인
3. 실제로 이 피처가 검증 스코어를 개선하는지, 베이스라인과 같은 조건으로 학습해서 비교

> **참고**: 이 환경은 CPU 1코어라 베이스라인과 동일한 `n_estimators=100`으로 두 모델을 다 돌리면 시간이 너무 오래 걸려서, 아래 비교는 `n_estimators=50`으로 축소했습니다. 상대적인 비교(개선 여부)에는 문제없지만, **최종 제출 전엔 원래 환경에서 `n_estimators=100`으로 다시 확인**하시는 걸 추천드려요.

## 7-1. `game_type`(F/R) — test에도 있는지 확인

`test.csv` 배포 샘플은 5행뿐이라 완전히 확신할 순 없지만, train에서 F가 특정 시즌 초반에만 몰린 이벤트성 값이 아니라 **매 시즌 3~10월 내내, 시즌마다 9.5~12.3% 비율로 꾸준히 등장**한다는 걸 먼저 확인합니다 — 즉 test(2025)에도 비슷한 비율로 나타날 가능성이 높다는 근거는 있지만, 5행 샘플이 전부 `R`이라 100% 확신하기는 어렵습니다.

In [10]:
test_sample = pd.read_csv(f"{DATA_DIR}/test.csv", encoding="utf-8-sig")
print(test_sample[["row_id", "season", "game_month", "game_type"]])
print()
print("=== train: 시즌별 game_type 비율 ===")
print(train.groupby("season")["game_type"].value_counts(normalize=True).unstack(fill_value=0).round(3))

        row_id  season  game_month game_type
0  TEST_000001    2025           7         R
1  TEST_000017    2025           3         R
2  TEST_000213    2025           6         R
3  TEST_005332    2025           3         R
4  TEST_035185    2025           3         R

=== train: 시즌별 game_type 비율 ===
game_type      F      R
season                 
2019       0.109  0.891
2020       0.095  0.905
2021       0.105  0.895
2022       0.123  0.877
2023       0.105  0.895
2024       0.118  0.882


5개 샘플이 우연히 전부 `R`일 수도 있는 정도의 표본이라, 이건 결론이 아니라 **열린 리스크로 남겨둡니다.** `game_type`은 뒤에서 보듯 중요도가 꽤 높게 나오는 피처라, test에 F가 거의 없거나 비율이 다르면 그만큼 흔들릴 수 있는 부분입니다.

## 7-2. 콜드스타트 플래그 통합 (16개 → 2개)

In [11]:
PITCHER_NA = [c for c in NA_COLS if c.startswith("asof_pitcher")]
BATTER_NA  = [c for c in NA_COLS if c.startswith("asof_batter")]

for df in (tr, val):
    df["pitcher_is_cold_start"] = df[PITCHER_NA].isna().any(axis=1).astype(int)
    df["batter_is_cold_start"]  = df[BATTER_NA].isna().any(axis=1).astype(int)

print(tr[["pitcher_is_cold_start", "batter_is_cold_start"]].mean())

pitcher_is_cold_start    0.000583
batter_is_cold_start     0.000652
dtype: float64


주의: 위 코드는 **콜드스타트 플래그를 만든 다음** median 대치를 해야 순서가 맞습니다 (이미 4번에서 median으로 채워버린 뒤라 이 노트북에서 재실행 시엔 4번 셀보다 앞으로 옮겨야 합니다 — 실제 파이프라인 코드에서는 아래 8번에서 순서를 바로잡았습니다).

## 7-3. 실제로 점수가 개선되는지 검증

같은 RandomForest 설정(depth=10, min_leaf=200)으로 (a) 베이스라인 피처 그대로 vs (b) 콜드스타트 플래그 2개 추가, 두 버전을 학습해서 val(2024) Brier Skill Score를 비교합니다.

In [12]:
from sklearn.ensemble import RandomForestClassifier

def eval_rf(Xtr, ytr, Xval, yval, n_estimators=50):
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=10,
                                 min_samples_leaf=200, n_jobs=1, random_state=42)
    rf.fit(Xtr, ytr)
    pred = rf.predict_proba(Xval)[:, 1]
    r = yval.mean(); brier = ((pred - yval) ** 2).mean(); base = r * (1 - r)
    score = max(0, 100000 * (1 - brier / base))
    return rf, brier, score

baseline_feats = feat_cols  # asof_* + 기본 피처 (콜드스타트 플래그 제외)
improved_feats = feat_cols + ["pitcher_is_cold_start", "batter_is_cold_start"]

_, b1, s1 = eval_rf(tr[baseline_feats], tr[TARGET], val[baseline_feats], val[TARGET])
print(f"[베이스라인 피처]        brier={b1:.6f}  score={s1:.2f}")

rf2, b2, s2 = eval_rf(tr[improved_feats], tr[TARGET], val[improved_feats], val[TARGET])
print(f"[+콜드스타트 플래그]     brier={b2:.6f}  score={s2:.2f}")

[베이스라인 피처]        brier=0.248796  score=404.88
[+콜드스타트 플래그]     brier=0.248761  score=418.52


In [13]:
imp = sorted(zip(improved_feats, rf2.feature_importances_), key=lambda x: -x[1])[:10]
for name, v in imp:
    print(f"{name:45s} {v:.4f}")

asof_pitcher_success_rate                    0.1473
game_type                                    0.1349
asof_pitcher_reverse_rate                    0.0907
asof_pitcher_prev5_game_success_rate         0.0796
asof_batter_success_rate                     0.0665
asof_pitcher_prev3_game_success_rate         0.0556
season                                       0.0526
asof_pitcher_prev1_game_success_rate         0.0459
asof_pitcher_middle_rate                     0.0261
asof_batter_middle_rate                      0.0259


## 결과 정리

- **(트리 수 50개, 상대비교 기준)** 콜드스타트 플래그 추가만으로 `score 404.88 → 418.52` (+13.64). 방향은 맞지만 아직 크지 않은 개선이에요 — 콜드스타트 행 자체가 전체의 0.06%(투수)/0.07%(타자)뿐이라 당연한 결과이기도 합니다. 효과를 더 보려면 '완전 콜드스타트(n=0)'뿐 아니라 **'표본이 적어서 불안정한'** 구간(예: `asof_pitcher_n < 50`)까지 신뢰도 가중치나 스무딩으로 다루는 게 나을 수 있습니다.
- **`game_type`이 2번째로 중요한 피처**로 나왔습니다. 상관관계 분석 때 예상했던 대로인데, 이게 test에서도 똑같이 작동할지는 7-1에서 봤듯 아직 확신할 수 없는 리스크입니다. 리더보드 제출 후 실제 점수와 로컬 val 점수 차이가 크게 벌어진다면 이 피처가 의심 후보 1순위입니다.
- `season`도 상위권(5위, 0.0526)에 있습니다 — 이건 지난 EDA에서 본 시즌별 성공률 하락 추세를 모델이 학습한 것인데, test는 2025년(train에 없던 값)이라 **외삽(extrapolation)** 이 됩니다. 트리 기반 모델은 학습 범위 밖 값에 대해 그냥 가장 가까운 리프의 값을 반환하므로, 2025년을 2024년과 거의 동일하게 취급할 가능성이 높습니다 — 하락 추세가 2025에도 이어진다면 살짝 과소보정될 수 있어요.

## 다음 단계 제안
1. **`n_estimators=100`으로 원래 환경에서 재검증** — 여기서는 CPU 제약으로 50트리만 비교했습니다.
2. `asof_pitcher_n`처럼 표본 크기가 작은 구간 전체를 신뢰도 가중(예: 베이지안 스무딩)으로 다뤄서 콜드스타트 효과를 더 키우기
3. `game_type`을 뺐을 때/넣었을 때 val 점수 변화 비교 — 만약 큰 차이가 없다면 안전하게 유지, 크게 의존하고 있다면 리스크로 판단하고 비중을 낮추는 것도 고려
4. `season`을 그대로 수치 피처로 쓰는 대신, `season`을 뺀 모델과 비교하거나 최근 시즌에 가중치를 주는 학습 방식(`sample_weight`) 검토
5. `trackman_history.csv` 활용한 피처 확장 (아직 미착수)

---
# 8. 모델 업그레이드: HistGradientBoosting

RandomForest보다 부스팅 계열이 이런 대용량 정형데이터 + Brier 최적화에 보통 더 유리합니다. 이 환경엔 인터넷이 없어서 LightGBM/XGBoost는 설치가 안 되지만, **sklearn 내장 `HistGradientBoostingClassifier`** 로 먼저 테스트합니다 (실제 제출 환경은 `requirements.txt`로 LightGBM/XGBoost 설치 가능 — 패키지 설치 단계에서는 인터넷이 열려 있다고 대회 가이드에 명시돼 있습니다).

In [14]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=6, l2_regularization=1.0,
    random_state=42, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
)
hgb.fit(tr[improved_feats], tr[TARGET])

pred = hgb.predict_proba(val[improved_feats])[:, 1]
r = val[TARGET].mean(); brier = ((pred - val[TARGET]) ** 2).mean(); base = r * (1 - r)
score = max(0, 100000 * (1 - brier / base))
print(f"[HGB, median대치+인코딩] n_iter={hgb.n_iter_}  brier={brier:.6f}  score={score:.2f}")

[HGB, median대치+인코딩] n_iter=300  brier=0.248525  score=513.23


RandomForest(50트리) 대비 **+95~112점** — 지금까지 시도한 것 중 가장 큰 개선폭입니다. `n_iter_=300`이 max_iter 그대로라 조기종료가 안 걸렸다는 뜻 = 아직 여지가 남아있다는 신호입니다.

## 8-1. HGB의 native NaN/카테고리 처리와 비교

HGB는 결측치를 그대로 넣어도 되고(분기 방향을 학습), 카테고리형도 `categorical_features`로 네이티브 지원합니다. 우리가 직접 만든 median 대치 + OrdinalEncoder 버전과 비교해봅니다.

In [15]:
for c in CAT_COLS:
    cats = tr[c].astype("category").cat.categories
    tr[c] = pd.Categorical(tr[c], categories=cats)
    val[c] = pd.Categorical(val[c], categories=cats)

cat_idx = [improved_feats.index(c) for c in CAT_COLS]
hgb_native = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.05, max_depth=6, l2_regularization=1.0,
    random_state=42, categorical_features=cat_idx,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
)
hgb_native.fit(tr[improved_feats], tr[TARGET])
pred2 = hgb_native.predict_proba(val[improved_feats])[:, 1]
brier2 = ((pred2 - val[TARGET]) ** 2).mean()
score2 = max(0, 100000 * (1 - brier2 / base))
print(f"[HGB, native NaN+카테고리] n_iter={hgb_native.n_iter_}  brier={brier2:.6f}  score={score2:.2f}")

[HGB, native NaN+카테고리] n_iter=400  brier=0.248572  score=494.34


**네이티브 방식이 오히려 더 낮게 나왔습니다** (494 vs 513). HGB의 자동 처리가 항상 우월한 건 아니라는 뜻이라, 직접 만든 median 대치 + OrdinalEncoder 버전을 기준으로 계속 갑니다.

## 8-2. Isotonic 캘리브레이션

Brier은 확률 자체의 정확도에 민감하므로, 후처리 캘리브레이션을 시도합니다. tr을 다시 학습(85%)/캘리브레이션(15%)으로 나눠서 진행합니다.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

tr_fit, tr_cal = train_test_split(tr, test_size=0.15, random_state=42, stratify=tr[TARGET])

hgb2 = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=6, l2_regularization=1.0,
    random_state=42, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
)
hgb2.fit(tr_fit[improved_feats], tr_fit[TARGET])

raw_val_pred = hgb2.predict_proba(val[improved_feats])[:, 1]
cal_pred_raw  = hgb2.predict_proba(tr_cal[improved_feats])[:, 1]

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(cal_pred_raw, tr_cal[TARGET])
cal_val_pred = iso.predict(raw_val_pred)

b0 = ((raw_val_pred - val[TARGET]) ** 2).mean(); s0 = max(0, 100000*(1-b0/base))
b1 = ((cal_val_pred - val[TARGET]) ** 2).mean(); s1 = max(0, 100000*(1-b1/base))
print(f"[캘리브레이션 전] brier={b0:.6f}  score={s0:.2f}")
print(f"[캘리브레이션 후] brier={b1:.6f}  score={s1:.2f}")

[캘리브레이션 전] brier=0.248526  score=512.83
[캘리브레이션 후] brier=0.248574  score=493.45


**캘리브레이션이 오히려 점수를 깎았습니다** (512.83 → 493.45). tr_cal이 2019~2023 중 랜덤 15%라서 val(2024)과 시즌 분포가 다른 게 원인으로 의심됩니다 — 지난 EDA에서 본 시즌별 성공률 하락 추세 때문에, 과거 데이터로 학습한 캘리브레이션 매핑이 2024(더 낮은 성공률)엔 어긋나게 적용됐을 가능성이 있습니다. **캘리브레이션 셋을 2024와 가장 가까운 시즌(2023)만으로 다시 구성**하면 결과가 달라질 수 있어 다음 단계로 남겨둡니다.

---
# 9. 1순위 시도: 시즌 맞춤 캘리브레이션 (결과: 보류)

아이디어: 캘리브레이션 셋을 랜덤 15%가 아니라 **val(2024)과 가장 가까운 2023시즌**으로 구성하면 캘리브레이션 매핑이 더 잘 맞지 않을까?

→ 이걸 하려면 2023을 학습에서 빼야 하는데, **그 자체로 원본 성능이 무너집니다.** 결과를 보면 이유가 명확합니다.

In [17]:
is_cal = tr["season"] == 2023
tr_train, tr_cal = tr.loc[~is_cal], tr.loc[is_cal]

hgb_nocal = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6,
                                            l2_regularization=1.0, random_state=42,
                                            early_stopping=True, validation_fraction=0.1, n_iter_no_change=20)
hgb_nocal.fit(tr_train[improved_feats], tr_train[TARGET])

raw_val = hgb_nocal.predict_proba(val[improved_feats])[:, 1]
b0 = ((raw_val - val[TARGET]) ** 2).mean(); s0 = max(0, 100000 * (1 - b0 / base))
print(f"[2019~2022만 학습, 2023 제외] brier={b0:.6f}  score={s0:.2f}")

cal_pred = hgb_nocal.predict_proba(tr_cal[improved_feats])[:, 1]
iso2 = IsotonicRegression(out_of_bounds="clip")
iso2.fit(cal_pred, tr_cal[TARGET])
cal_val = iso2.predict(raw_val)
b1 = ((cal_val - val[TARGET]) ** 2).mean(); s1 = max(0, 100000 * (1 - b1 / base))
print(f"[2023으로 캘리브레이션 후]     brier={b1:.6f}  score={s1:.2f}")

[2019~2022만 학습, 2023 제외] brier=0.255583  score=0.00
[2023으로 캘리브레이션 후]     brier=0.249672  score=54.16


**2023을 학습에서 빼는 순간 score가 513 → 0으로 무너집니다.** 캘리브레이션으로 일부 복구되긴 하지만(0→54) 애초에 잃은 게 훨씬 커서 손해예요. → **이 데이터에서는 '최근 시즌'을 캘리브레이션 재료로 아껴두는 것보다 학습에 직접 넣는 게 압도적으로 이득**입니다. 시즌 드리프트가 생각보다 강하다는 걸 재확인했습니다.

(캘리브레이션 자체를 완전히 포기하는 건 아니고, 학습 데이터를 전부 쓰면서 K-fold out-of-fold 예측으로 캘리브레이션하는 방식이면 이 트레이드오프를 피할 수 있습니다 — 모델을 여러 번 학습해야 해서 비용이 커 다음 기회로 미룹니다.)

→ **1순위 보류, 2순위(하이퍼파라미터 튜닝)로 이동.**

# 10. 2순위: HGB 하이퍼파라미터 튜닝

`max_depth`, `learning_rate`, `max_iter`, `l2_regularization` 조합을 그리드로 비교합니다.

In [18]:
def eval_hgb(**kwargs):
    hgb = HistGradientBoostingClassifier(random_state=42, early_stopping=True,
                                          validation_fraction=0.1, n_iter_no_change=20, **kwargs)
    hgb.fit(tr[improved_feats], tr[TARGET])
    pred = hgb.predict_proba(val[improved_feats])[:, 1]
    b = ((pred - val[TARGET]) ** 2).mean()
    s = max(0, 100000 * (1 - b / base))
    return hgb.n_iter_, b, s

grid = [
    dict(name="depth6_lr0.05_iter300 (기본)", max_iter=300, learning_rate=0.05, max_depth=6, l2_regularization=1.0),
    dict(name="depth6_lr0.03_iter600",        max_iter=600, learning_rate=0.03, max_depth=6, l2_regularization=1.0),
    dict(name="depth8_lr0.05_iter300",        max_iter=300, learning_rate=0.05, max_depth=8, l2_regularization=1.0),
    dict(name="depth10_lr0.05_iter300",       max_iter=300, learning_rate=0.05, max_depth=10, l2_regularization=1.0),
    dict(name="depth8_lr0.03_iter600",        max_iter=600, learning_rate=0.03, max_depth=8, l2_regularization=1.0),
    dict(name="depth8_lr0.05_iter500",        max_iter=500, learning_rate=0.05, max_depth=8, l2_regularization=1.0),
    dict(name="depth8_lr0.05_iter300_l2=3",   max_iter=300, learning_rate=0.05, max_depth=8, l2_regularization=3.0),
]
for cfg in grid:
    name = cfg.pop("name")
    n_iter, b, s = eval_hgb(**cfg)
    print(f"[{name:32s}] n_iter={n_iter:3d}  brier={b:.6f}  score={s:.2f}")

[depth6_lr0.05_iter300 (기본)   ] n_iter=300  brier=0.248525  score=513.23
[depth6_lr0.03_iter600          ] n_iter=600  brier=0.248551  score=502.79
[depth8_lr0.05_iter300          ] n_iter=300  brier=0.248368  score=576.08
[depth10_lr0.05_iter300         ] n_iter=300  brier=0.248450  score=543.18
[depth8_lr0.03_iter600          ] n_iter=600  brier=0.248456  score=540.92
[depth8_lr0.05_iter500          ] n_iter=500  brier=0.248549  score=503.37
[depth8_lr0.05_iter300_l2=3     ] n_iter=300  brier=0.248426  score=552.99


## 결과

**`max_depth=8, learning_rate=0.05, max_iter=300, l2_regularization=1.0`이 최고 — score 576.08.**

- `depth=8`이 제일 큰 레버였고, 그 이상(`depth=10`)이나 `iter`를 더 늘리는 건 오히려 손해였습니다.
- 특히 `iter=500`(조기종료 없이 끝까지)이 `iter=300`보다 나빴다는 게 흥미롭습니다 — HGB의 내부 early-stopping은 tr에서 랜덤 10%를 떼어 판단하는데, 이 10%는 2019~2023이 섞인 분포라 val(2024)의 최근 트렌드와는 안 맞을 수 있습니다. 여기서도 시즌 드리프트가 은근히 영향을 주고 있는 것으로 보입니다.

## 지금까지 전체 그림
```
RF(50트리) + 콜드스타트         →  401
HGB 기본(depth=6)               →  513   (+112, 모델 교체 효과)
HGB 튜닝(depth=8)                →  576   (+63,  하이퍼파라미터 효과)
```

## 다음 단계
1. `game_type` 의존도 점검 (빼고 학습 vs 넣고 학습 비교)
2. `season` 외삽 문제 — 빼보기 / `sample_weight`로 최근 시즌 가중
3. K-fold out-of-fold 캘리브레이션 (학습 데이터 낭비 없이 캘리브레이션)
4. `trackman_history.csv` 피처 확장
5. 실제 제출 환경에서 LightGBM/XGBoost + 더 넓은 그리드서치